# Example 03 Lecture Note: Image Bifiltration (Distance + Intensity)

This notebook demonstrates image-native ingestion and featurization.

## Learning goals
- Build deterministic synthetic image classes with masks.
- Encode with `ImageDistanceBifiltration`.
- Inspect direct invariants before vectorization.
- Export a feature table for downstream modeling.
        


## Modeling decisions and alternatives

Chosen here:
- one mask per image (`ImageDistanceBifiltration(mask=...)`)
- `degree=0` and `field=F2()` for robust first-pass runtime
- feature composition: Euler surface + sliced barcode summaries

Alternatives:
- lower-star/cubical-only image filtrations when distance channel is not needed
- landscapes instead of sliced barcode summaries
- stricter PL mode (`pl_mode=:verified`) if runtime budget allows
        


In [ ]:
using Random
using SparseArrays
using Statistics
using Printf

repo_root = if isdir(joinpath(pwd(), "src"))
    pwd()
elseif isdir(joinpath(pwd(), "..", "src"))
    normpath(joinpath(pwd(), ".."))
else
    pwd()
end

try
    using PosetModules
catch
    include(joinpath(repo_root, "src", "PosetModules.jl"))
    using .PosetModules
end
const PM = PosetModules

Random.seed!(20260217)
outdir = joinpath(repo_root, "examples", "_outputs", "03_image_bifiltration_notebook")
mkpath(outdir)
println("repo_root = ", repo_root)
println("outdir    = ", outdir)


## Step 1: Build a deterministic synthetic image dataset


In [ ]:
function make_image_distance_dataset(; n_per_class::Int=5, side::Int=40, seed::Int=3003)
    rng = MersenneTwister(seed)
    imgs = PM.ImageNd[]
    masks = BitMatrix[]
    labels = String[]

    for _ in 1:n_per_class
        img = Matrix{Float64}(undef, side, side)
        cx = side / 2 + 0.8 * randn(rng)
        cy = side / 2 + 0.8 * randn(rng)
        @inbounds for y in 1:side, x in 1:side
            dx = (x - cx) / side
            dy = (y - cy) / side
            r = sqrt(dx * dx + dy * dy)
            img[y, x] = exp(-((r - 0.22)^2) / 0.004) + 0.03 * randn(rng)
        end
        push!(imgs, PM.ImageNd(img))
        push!(masks, img .> quantile(vec(img), 0.62))
        push!(labels, "annulus")
    end

    for _ in 1:n_per_class
        img = Matrix{Float64}(undef, side, side)
        c1x = 0.34 * side + 0.8 * randn(rng)
        c1y = 0.50 * side + 0.8 * randn(rng)
        c2x = 0.66 * side + 0.8 * randn(rng)
        c2y = 0.50 * side + 0.8 * randn(rng)
        @inbounds for y in 1:side, x in 1:side
            g1 = exp(-(((x - c1x)^2 + (y - c1y)^2) / (2 * 3.5^2)))
            g2 = exp(-(((x - c2x)^2 + (y - c2y)^2) / (2 * 3.5^2)))
            img[y, x] = g1 + g2 + 0.03 * randn(rng)
        end
        push!(imgs, PM.ImageNd(img))
        push!(masks, img .> quantile(vec(img), 0.70))
        push!(labels, "two_blobs")
    end

    return imgs, masks, labels
end

imgs, masks, labels = make_image_distance_dataset(n_per_class=5, side=40, seed=3003)
println("dataset size = ", length(imgs))
println("mask shape   = ", size(masks[1]))


## Step 2: Encode each image


In [ ]:
field = PM.CoreModules.F2()
encodings = Vector{PM.EncodingResult}(undef, length(imgs))
for i in eachindex(imgs)
    filt = PM.ImageDistanceBifiltration(; mask=masks[i])
    encodings[i] = PM.encode(imgs[i], filt; degree=0, field=field, cache=:auto)
end

println("encoded images = ", length(encodings))
println("one enc type   = ", typeof(encodings[1]))


## Step 3: Inspect direct invariants before featurization


In [ ]:
opts = PM.InvariantOptions(
    ; axes_policy=:encoding, max_axis_len=64,
      threads=true, strict=true, pl_mode=:fast,
)

surf0 = PM.euler_surface(encodings[1]; opts=opts)
println("size(euler_surface first sample) = ", size(surf0))

dirs = [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]
offs = [
    collect(range(-0.5, stop=0.5, length=5)),
    collect(range(-0.5, stop=0.5, length=5)),
    collect(range(-0.7, stop=0.7, length=5)),
]

sb0 = PM.slice_barcodes(encodings[1]; opts=opts, directions=dirs, offsets=offs, threads=true)
println("slice_barcodes key count = ", length(keys(sb0)))


## Step 4: Feature specification and batch transform


In [ ]:
euler_spec = PM.EulerSurfaceSpec(; axes_policy=:encoding, max_axis_len=64, threads=true)
sbar_spec = PM.SlicedBarcodeSpec(
    ; directions=dirs, offsets=offs, featurizer=:summary,
      aggregate=:mean, normalize_weights=true, threads=true,
)
spec = PM.CompositeSpec((euler_spec, sbar_spec))

samples = [(; M=encodings[i].M, pi=encodings[i].pi, label=labels[i], id=@sprintf("img_%03d", i)) for i in eachindex(encodings)]

sc = PM.SessionCache()
fs = PM.batch_transform(
    samples,
    spec;
    opts=opts,
    idfun=s -> s.id,
    labelfun=s -> s.label,
    batch=PM.BatchOptions(threaded=true, backend=:threads, progress=false, deterministic=true),
    cache=sc,
)

println("feature matrix shape = ", size(fs.X))
println("first 5 feature names= ", PM.feature_names(spec)[1:5])


## Step 5: Export outputs


In [ ]:
wide_path = joinpath(outdir, "image_distance_bifiltration__wide.csv")
long_path = joinpath(outdir, "image_distance_bifiltration__long.csv")

PM.save_features(wide_path, fs; format=:csv, mode=:wide, metadata=true)
PM.save_features(long_path, fs; format=:csv, mode=:long, metadata=true)

println("wide csv = ", wide_path)
println("long csv = ", long_path)


## Discussion prompts
- How sensitive are results to mask threshold choice?
- Which information is captured by Euler but missed by slice summaries (or vice versa)?
- When would you increase `max_axis_len`, and what runtime cost would you expect?
        
